In [ ]:
from trustutils import run
run.TRUST_parameters()
run.introduction('MMS Diffusion 3D', '2026')


In [ ]:
run.reset()

NB_PROCS = 2

case = run.addCase('.', 'MMS_Diff3D.data', nbProcs=NB_PROCS)
case.partition()          # trust -partition MMS_Diff3D.data 2 -> PAR_MMS_Diff3D.data + DOM_000*.Zones
run.printCases()
run.runCases()
run.tablePerf()


In [ ]:
from trustutils.files import SonFile
import numpy as np

BUILD = run.BUILD_DIRECTORY

def read_son(case, name):
    """Return (time, values) for probe `name` of case `case` (dataset name)."""
    path = f"{BUILD}/{case}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

# MMS_Diff3D: b(x, y, 0) = sin(pi y) with Dirichlet zero at y = 0, 1 and
# Neumann walls elsewhere is an exact eigenmode of the diffusion operator:
#     b(x, y, t) = sin(pi y) exp(-eta pi^2 t),  eta = 0.05.
# All three probes sit on the mode, so each must follow the same analytic
# decay.  The expected ratio uses the ACTUAL final time of the run.
eta = 0.05

for name in ('SONDE_Y025', 'SONDE_Y050', 'SONDE_Y075'):
    t, b = read_son(case.dataFileName_, name)
    ratio = b[-1] / b[0]
    expected = np.exp(-eta * np.pi**2 * t[-1])
    err = abs(ratio - expected) / expected
    print(f"{name}: ratio={ratio:.4f} expected={expected:.4f} err={err:.3e}")
    assert err < 0.10, f"{name} decay error too large: {err:.3e}"
print("OK: sin(pi y) mode decays at the analytic eta pi^2 rate at all probes.")
